In [17]:
import json
import os

*load different configs*

In [18]:
def load_risk_config():
    """Load risk statistics from JSON config"""
    config_path = os.path.join('../../configs/risk_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)

def load_extraction_config():
    """Load extraction decision rules from JSON config"""
    config_path = os.path.join('../../configs/extraction_type_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)

def load_binary_config():
    """Load binary removal decision config from JSON"""
    config_path = os.path.join('../../configs/extraction_binary_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)
    
def load_client_profiles():
    """Load client profiles from JSON config"""
    config_path = os.path.join('../../configs/client_profiles.json')
    with open(config_path, 'r') as f:
        profiles = json.load(f)
    # Convert client ID keys to integers
    result = {}
    for client_id, profile in profiles.items():
        profile_copy = profile.copy()
        # Convert score_scale keys from strings to integers
        if 'score_scale' in profile_copy:
            profile_copy['score_scale'] = {int(k): v for k, v in profile_copy['score_scale'].items()}
        result[int(client_id)] = profile_copy
    return result

def load_noise_config():
    """Load measurement noise configuration"""
    config_path = os.path.join('../../configs/noise_config.json')
    with open(config_path, 'r') as f:
        return json.load(f)

def load_generation_config():
    """Load dataset generation parameters"""
    config_path = os.path.join('../../configs/generation_config.json')
    with open(config_path, 'r') as f:
        return json.load(f)

# Load all configs
CLIENT_PROFILES = load_client_profiles()
NOISE_CONFIG = load_noise_config()
GEN_CONFIG = load_generation_config()

In [19]:
import numpy as np
import pandas as pd

np.random.seed(GEN_CONFIG['dataset']['random_seed'])

# Helper functions
def generate_binary(n, p):
    return np.random.choice([0, 1], size=n, p=[1-p, p])

def generate_categorical(n, categories, probs):
    return np.random.choice(categories, size=n, p=probs)

def generate_age(n, mu=28, sigma=7, lo=16, hi=60):
    return np.clip(np.random.normal(loc=mu, scale=sigma, size=n).astype(int), lo, hi)

def softmax(vec, temperature=0.8):
    v = np.array(list(vec), dtype=float)
    v = v - v.max()
    ex = np.exp(v / max(1e-6, temperature))
    return ex / ex.sum()

In [ ]:
def get_profile(client_id):
    return CLIENT_PROFILES.get(client_id,
        {"name": f"Clinic_{client_id}",
         "prevalence_shift": {},
         "score_scale": {1:1,2:1,3:1,4:1},
         "missingness": {}})

# global sizes
n_clients = GEN_CONFIG['dataset']['n_clients']
patients_per_client = GEN_CONFIG['dataset']['patients_per_client']

rows = []

# Generate per client

for c in range(1, n_clients+1):
    prof = get_profile(c)
    n = patients_per_client

    # Prevalence shifts / distributions
    Age_mu = prof["prevalence_shift"].get("Age_mu", 28)
    Proximity_p = prof["prevalence_shift"].get("Proximity_Nerve_p", 0.60)
    Depth_probs = prof["prevalence_shift"].get("Impaction_Depth", [0.45,0.45,0.1])

    # Features
    age = generate_age(n, mu=Age_mu)
    sex = generate_binary(n, 0.5)
    mandi_maxi= generate_binary(n, 0.48) # 0 mandi, 1 maxi

    pain = generate_binary(n, 0.5)
    swelling = generate_binary(n, 0.30)
    trismus = generate_binary(n, 0.20)
    pericoronitis = generate_binary(n, 0.75)

    caries_w = generate_binary(n, 0.20)
    caries_adj = generate_binary(n, 0.15)
    perio = generate_categorical(n, [1,2,3], [0.6,0.3,0.1]) # 1 healthy, 3 severe
    root_dev = generate_categorical(n, [1,2,3], [0.2,0.7,0.1]) # 1 incomplete, 2 complete, 3 complex/ 
    mobility = generate_categorical(n, [0,1,2], [0.7,0.2,0.1]) # 0 none, 2 severe

    # Angulation: 1 vertical, 2 mesioangular, 3 distoangular, 4 horizontal, 5 transverse (very rare)
    ang = generate_categorical(n, [1, 2, 3, 4, 5], [0.30, 0.40, 0.20, 0.09, 0.01])

    # Depth: 1 soft tissue, 2 partial bony, 3 deep bony
    depth = generate_categorical(n, [1, 2, 3], Depth_probs)

    # Gingival coverage subcategory for partial bony (depth == 2)
    # 0 = fully_covered, 1 = partially_covered (60%)
    gingival_cov = np.full(n, np.nan)  # default NaN for non-depth == 2
    mask_depth2 = (depth == 2)
    gingival_cov[mask_depth2] = generate_binary(mask_depth2.sum(), p=0.60) # 60% partially covered

    prox_nerve = generate_binary(n, Proximity_p) # 1 = close/uncertain to IAN
    root_count = generate_categorical(n, [1,2,3,4], [0.1,0.5,0.3,0.1])
    root_curve = generate_binary(n, 0.70) # 1 = curved/divergent
    bone_density = generate_categorical(n, [1,2,3], [0.4,0.4,0.2]) # 3 = high

    cyst = generate_binary(n, 0.10)

    diabetes = generate_binary(n, 0.10)
    clotting = generate_binary(n, 0.02)
    smoking = generate_binary(n, 0.25)
    prev_issue = generate_binary(n, 0.10)

    # Osteo and Biophos
    # Age/sex-conditional osteoporosis
    def p_osteoporosis(a, s):
        # s: 0=male, 1=female
        if s == 0:  # male
            if a < 40: return 0.002
            if a < 50: return 0.010
            return 0.030
        else:       # female
            if a < 40: return 0.005
            if a < 50: return 0.020
            return 0.070

    osteoporosis = np.array([np.random.rand() < p_osteoporosis(age[i], sex[i]) for i in range(n)], dtype=int)

    # --- Bisphosphonates conditional on osteoporosis (+ mild age effect)
    def p_bisphosph(a, has_osteo):
        if has_osteo:
            base = 0.45
            # a bit higher uptake in older patients
            if a >= 50: base += 0.10
            return min(base, 0.70)  # cap
        else:
            return 0.003  # rare non-osteoporosis indications in this cohort

    bisphosph = np.array([np.random.rand() < p_bisphosph(age[i], osteoporosis[i]) for i in range(n)], dtype=int)

    for i in range(n):
        rows.append({
            "Client": c, "Patient": i+1,
            "Age": age[i], "Sex": sex[i], "Mandi_Maxi": mandi_maxi[i],
            "Pain": pain[i], "Swelling": swelling[i], "Trismus": trismus[i], "Pericoronitis": pericoronitis[i],
            "Caries_Wisdom": caries_w[i], "Caries_Adjacent": caries_adj[i],
            "Periodontal_Status": perio[i], "Root_Development": root_dev[i], "Tooth_Mobility": mobility[i],
            "Tooth_Angulation": ang[i], "Impaction_Depth": depth[i], "PartialBony_GingivalCoverage": (None if np.isnan(gingival_cov[i]) else int(gingival_cov[i])),
            "Proximity_Nerve": prox_nerve[i],
            "Root_Count": root_count[i], "Root_Curvature": root_curve[i], "Bone_Density": bone_density[i],
            "Cyst": cyst[i],
            "Diabetes": diabetes[i], "Osteoporosis": osteoporosis[i], "Clotting_Disorder": clotting[i],
            "Smoking": smoking[i], "Bisphosphonates": bisphosph[i],
            "Prev_Extraction_Issue": prev_issue[i],
        })

df = pd.DataFrame(rows)

# Optional: light missingness per profile
for c in range(1, n_clients+1):
    miss = get_profile(c)["missingness"]
    idx = df["Client"] == c
    for col, rate in miss.items():
        mask = (np.random.rand(idx.sum()) < rate)
        df.loc[idx, col] = df.loc[idx, col].mask(mask)

In [21]:
# Common Utility Functions to apply configs

import math
import numpy as np

def _norm_key(v):
    """Normalize row values so they match JSON keys like '0','1','2', floats sometimnes got interpretet as NaN"""
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    # Booleans first
    if isinstance(v, (bool, np.bool_)):
        return "1" if v else "0"
    # Integers stay integers
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    # Floats that are really integers (0.0, 1.0) -> '0','1'
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer():
            return str(int(v))
        # else allow exact float string (rare for your use case)
        return str(v)
    # Fallback
    return str(v)

def safe_get_value(row, feature):
    """Safely get feature value from row, handling missing/None values"""
    if feature not in row or row[feature] is None:
        return None
    return row[feature]


def get_modifier_for_value(modifiers, value):
    """
    Get modifier multiplier for a given value from a modifiers dict.
    
    Handles both:
    - Simple dict: {"1": 2.0, "2": 1.5}
    - Age range list: [{"min": 12, "max": 24, "mult": 1.1}, ...]
    """
    if not modifiers:
        return 1.0
    
    # Handle age ranges (list of {min, max, mult})
    if isinstance(modifiers, list):
        value = int(value) if value is not None and not (isinstance(value,float) and math.isnan(value)) else None
        if value is not None:
            for age_range in modifiers:
                if age_range["min"] <= value <= age_range["max"]:
                    return age_range["mult"]
        return 1.0
    
    # Handle regular dict lookups
    key = _norm_key(value)
    if key is not None and key in modifiers:
        return modifiers[key]
    return 1.0


def check_condition_match(row, feature, expected_values):
    """
    Check if a feature's value in the row matches expected values.
    
    Handles:
    - Simple value: "2"
    - List of values: [2, 3]
    - Age ranges: {"min": 25, "max": 34}
    - Comparison operators: ">=3" (handles ">=", "<=", etc.)
    """
    if feature not in row or row[feature] is None or (isinstance(row[feature], float) and math.isnan(row[feature])):
        return False
    val_key = _norm_key(row[feature])

    # Handle dict with min/max (age ranges in conditions)
    if isinstance(expected_values, dict) and "min" in expected_values and "max" in expected_values:
        v = row[feature]
        return expected_values["min"] <= v <= expected_values["max"]
    
    # Handle list of values
    # list of values (may contain '>=3', etc.)
    if isinstance(expected_values, list) and len(expected_values) > 0:
        first = expected_values[0]
        if isinstance(first, str) and any(op in first for op in [">=", "<=", ">", "<", "==", "!="]):
            s = first
            v = row[feature]
            if ">=" in s: return v >= int(s.replace(">=",""))
            if "<=" in s: return v <= int(s.replace("<=",""))
            if ">" in s: return v >  int(s.replace(">",""))
            if "<" in s: return v <  int(s.replace("<",""))
            if "==" in s: return v == int(s.replace("==",""))
            if "!=" in s: return v != int(s.replace("!=",""))
        # normal membership, normalize each expected to same keyspace
        expected_keys = {_norm_key(ev) for ev in expected_values}
        return val_key in expected_keys

    # single value
    return val_key == _norm_key(expected_values)


def apply_modifiers_to_value(base_value, modifiers_dict, row, feature_name):
    """
    Apply modifiers to a base value based on row data.
    
    This is the core logic for applying any type of modifier.
    Returns the multiplier to apply (1.0 if no match).
    """
    value = safe_get_value(row, feature_name)
    if value is None:
        return 1.0
    
    return get_modifier_for_value(modifiers_dict, value)

In [22]:
# compute_risk_from_evidence

def compute_risk_from_evidence(row, risk_type, risk_config, surgical_decision=None):
    """
    Compute risk based on evidence table configuration.
    """
    risk_data = risk_config[risk_type]
    risk = risk_data["base_incidence"]
    
    # Apply risk modifiers using unified approach
    for feature, modifiers in risk_data["risk_modifiers"].items():
        mult = apply_modifiers_to_value(1.0, modifiers, row, feature)
        risk *= mult
    
    # Apply interactions using unified condition matching
    for interaction in risk_data.get("interactions", []):
        conditions = interaction["conditions"]
        if all(check_condition_match(row, f, v) for f, v in conditions.items()):
            risk *= interaction["mult"]
    
    # Apply surgery modifiers
    if surgical_decision is not None and "surgery_modifiers" in risk_data:
        mult = apply_modifiers_to_value(1.0, risk_data["surgery_modifiers"], 
                                       {"surgery": surgical_decision}, "surgery")
        risk *= mult
    
    # Apply domain-specific hard gates
    if risk_type == "NerveDysesthesia":
        # Gate 1: Clamp down if IAN not close
        if safe_get_value(row, "Proximity_Nerve") == 0:
            risk *= 0.30
        # Gate 2: Zero risk for maxillary teeth
        if safe_get_value(row, "Mandi_Maxi") == 1:
            risk = 0.0
    
    return min(1.0, max(0.0, risk))


In [23]:
# compute_removal_decision

def compute_removal_decision(row):
    """
    Binary decision: Should we remove this third molar at all? 
    
    REFACTORED VERSION using common utilities.
    Same functionality, cleaner code.
    """
    config = load_binary_config()
    removal_prob = config["base_prevalence"]
    
    # Apply positive indications
    for feature, modifiers in config["positive_indications"].items():
        mult = apply_modifiers_to_value(1.0, modifiers, row, feature)
        removal_prob *= mult
    
    # Apply counter-indications (special handling for Systemic_Risk nested structure)
    for feature, modifiers in config["counter_indications"].items():
        if feature == "Systemic_Risk":
            # Nested dict: each risk factor has its own modifiers
            for risk_factor, risk_modifiers in modifiers.items():
                mult = apply_modifiers_to_value(1.0, risk_modifiers, row, risk_factor)
                removal_prob *= mult
        else:
            # All other features use standard modifier logic
            mult = apply_modifiers_to_value(1.0, modifiers, row, feature)
            removal_prob *= mult
    
    # Apply contextual modifiers
    for feature, modifiers in config["contextual_modifiers"].items():
        mult = apply_modifiers_to_value(1.0, modifiers, row, feature)
        removal_prob *= mult
    
    # Apply interactions using unified condition matching
    for interaction in config.get("interactions", []):
        conditions = interaction["conditions"]
        match = all(
            check_condition_match(row, feature, expected_values)
            for feature, expected_values in conditions.items()
        )
        if match:
            removal_prob *= interaction["mult"]
    
    # Clamp and sample
    removal_prob = min(1.0, max(0.0, removal_prob))
    removal_decision = int(np.random.rand() < removal_prob)
    return removal_decision, removal_prob

In [24]:
# class semantics
# 1 = Simple extraction
# 2 = Surgical extraction (flap/bone)  -> has subtype
# 3 = Coronectomy (nerve-sparing, high IAN risk)
# evaluate_rule_conditions and apply_rule_category

def evaluate_rule_conditions(row, rule_data):
    """
    Evaluate rule conditions against patient data.
    """
    if "conditions" not in rule_data:
        return False
    
    conditions = rule_data["conditions"]
    # Check all conditions match
    if not all(check_condition_match(row, feature, expected_values) 
               for feature, expected_values in conditions.items()):
        return False
    
    # Check additional conditions if present
    if "additional_conditions" in rule_data:
        additional = rule_data["additional_conditions"]
        if not all(check_condition_match(row, feature, values) 
                   for feature, values in additional.items()):
            return False
    
    return True


def apply_rule_category(row, scores, rule_category, config):
    """Apply a category of rules to the scores."""
    if rule_category not in config:
        return scores
    
    for rule_name, rule_data in config[rule_category].items():
        # Check if nested structure (rule_name IS the feature: Tooth_Mobility, Root_Development, etc.)
        if isinstance(rule_data, dict) and not any(key in rule_data for key in ['conditions', 'effects', 'description']):
            # Nested: use patient's value to look up nested dict
            value = safe_get_value(row, rule_name)
            if value is not None and str(value) in rule_data:
                for cls, effect in rule_data[str(value)]["effects"].items():
                    add(scores, int(cls), effect)
        elif rule_category == "age_rules" and rule_name == "Age_Scaling":
            # Special: age scaling rule
            age = safe_get_value(row, "Age")
            if age is not None:
                age_factor = max(0.0, min(2.0, (age - 25) / 10.0))
                for cls, effect in rule_data["effects"].items():
                    add(scores, int(cls), effect * age_factor)
        elif evaluate_rule_conditions(row, rule_data):
            # Standard: flat structure with conditions
            for cls, effect in rule_data["effects"].items():
                add(scores, int(cls), effect)
    
    return scores

def get_base_priors():
    """Get base priors from extraction config"""
    config = load_extraction_config()
    return {int(k): v for k, v in config["_base_priors"].items()}

def add(scores, cls, amt):
    scores[cls] += amt

def compute_scores_row(row):
    # Get base priors from config
    base_priors = get_base_priors()
    s = {1: base_priors[1], 2: base_priors[2], 3: base_priors[3], 4: base_priors[4]}
    
    config = load_extraction_config()
    
    # Apply each rule category
    s = apply_rule_category(row, s, "symptom_rules", config)
    s = apply_rule_category(row, s, "anatomy_rules", config)
    s = apply_rule_category(row, s, "pathology_rules", config)
    s = apply_rule_category(row, s, "mobility_rules", config)
    s = apply_rule_category(row, s, "root_development_rules", config)
    s = apply_rule_category(row, s, "periodontal_rules", config)
    s = apply_rule_category(row, s, "caries_rules", config)
    s = apply_rule_category(row, s, "systemic_rules", config)
    s = apply_rule_category(row, s, "age_rules", config)
    s = apply_rule_category(row, s, "ian_proximity_rules", config)
    s = apply_rule_category(row, s, "symptom_interactions", config)
    
    # Apply client preferences
    scales = get_profile(int(row["Client"]))["score_scale"]
    for k in s:
        s[k] *= scales.get(k, 1.0)
    
    return s

In [25]:
def decide_row(row, temperature=None, noise_sd=None):
    # Load from config if not provided
    if temperature is None:
        temperature = GEN_CONFIG['decision_model']['temperature']
    if noise_sd is None:
        noise_sd = GEN_CONFIG['decision_model']['noise_sd']
    
    s = compute_scores_row(row)
    for k in s:
        s[k] += np.random.normal(0.0, noise_sd)
    probs = softmax([s[1], s[2], s[3], s[4]], temperature=temperature)
    decision = np.random.choice([1,2,3,4], p=probs)
    return decision, s, probs

## Individual variation for more complexity and non-deterministic value generation

In [26]:
def add_feature_noise(df, client_id):
    prof = get_profile(client_id)
    noise_level = prof.get("measurement_noise", 0.05)

    client_mask = df["Client"] == client_id

    # ordinal features, ADD MORE LATER ON
    ordinal_config = NOISE_CONFIG['ordinal_features']
    ordinal_features = {}
    for feat, params in ordinal_config.items():
        ordinal_features[feat] = (
            params['min'],
            params['max'],
            noise_level * params['noise_multiplier']
        )

    # apply noise to ordinal features
    for feat, (min_val, max_val, prob) in ordinal_features.items():
        mask = client_mask & (np.random.rand(len(df)) < prob)
        # randomly shift value down or up
        shift = np.random.choice([-1, 1], size=mask.sum())
        # apply shift and clip to valid range of feature (min_val, max_val)
        df.loc[mask, feat] = np.clip(df.loc[mask, feat] + shift, min_val, max_val)

    # binary features, ADD MORE LATER ON
    binary_config = NOISE_CONFIG['binary_features']
    binary_features = {}
    for feat, params in binary_config.items():
        binary_features[feat] = noise_level * params['noise_multiplier']
    
    # apply noise to binary features
    for feature, prob in binary_features.items():
        mask = client_mask & (np.random.rand(len(df)) < prob)
        # flip the binary value
        df.loc[mask, feature] = 1 - df.loc[mask, feature]
    
    return df

In [27]:
risk_config = load_risk_config()

decisions, score1, score2, score3, score4, p1, p2, p3, p4, subtypes = [], [], [], [], [], [], [], [], [], []

removal_decisions, removal_probs = [], []
alveolar_risks, infection_risks, nerve_risks, bleeding_risks = [], [], [], []

# apply the noise to the dataset before computing risks, so that the risks are not based on deterministic values
for c in range(1, n_clients+1):
    df = add_feature_noise(df, c)

for _, row in df.iterrows():
    # Primary surgical decision
    d, s, p = decide_row(row)
    decisions.append(d)
    score1.append(s[1]); score2.append(s[2]); score3.append(s[3]); score4.append(s[4])
    p1.append(p[0]); p2.append(p[1]); p3.append(p[2]); p4.append(p[3])
    
    # Surgical subtype
    if d == 2:
        depth = row["Impaction_Depth"]
        if depth == 1:
            subtypes.append(1)
        elif depth == 2:
            subtypes.append(2)
        else:
            subtypes.append(3)
    else:
        subtypes.append(None)
    
    # Removal decision
    removal_decision, removal_prob = compute_removal_decision(row)
    removal_decisions.append(removal_decision)
    removal_probs.append(removal_prob)
    
    # Risk assessment
    alveolar_risk = compute_risk_from_evidence(row, "AlveolarOsteitis", risk_config, d)
    infection_risk = compute_risk_from_evidence(row, "SecondaryInfection", risk_config, d)
    nerve_risk = compute_risk_from_evidence(row, "NerveDysesthesia", risk_config, d)
    bleeding_risk = compute_risk_from_evidence(row, "Bleeding", risk_config, d)
    
    alveolar_risks.append(alveolar_risk)
    infection_risks.append(infection_risk)
    nerve_risks.append(nerve_risk)
    bleeding_risks.append(bleeding_risk)

# Add all columns to dataframe
# Existing columns
df["Surgical_Extraction_Type"] = decisions
df["Score_1"] = score1; df["Score_2"] = score2; df["Score_3"] = score3; df["Score_4"] = score4
df["Prob_1"] = p1; df["Prob_2"] = p2; df["Prob_3"] = p3; df["Prob_4"] = p4
df["Surg_2_Subtype"] = subtypes

df["Removal_Indicated"] = removal_decisions
df["Removal_Prob"] = removal_probs
df["Risk_AlveolarOsteitis"] = alveolar_risks
df["Risk_SecondaryInfection"] = infection_risks
df["Risk_NerveDysesthesia"] = nerve_risks
df["Risk_Bleeding"] = bleeding_risks

In [28]:
# Load output config
output_config = GEN_CONFIG['output']
output_dir = output_config['output_dir']
filename_base = output_config['filename_base']

# Save files
df.to_excel(f"{output_dir}{filename_base}.xlsx", index=False)
df.to_csv(f"{output_dir}{filename_base}.csv", index=False)

print(f"Saved: {filename_base} to {output_dir}")
print(f"Dataset shape: {df.shape}")
print(f"Formats: {', '.join(output_config['formats'])}")

Saved: fed_recommenders_synthetic_dataset to ../../data/raw/
Dataset shape: (2000, 44)
Formats: csv, xlsx
